### 3 Model conversation:
Openai, Google and Ollama: discussing Marvel Movies

Note: At first created 3 model conversation method which was similar to the 2 model conversation code, had just added a new model to the code 
with few changes, it worked but the conversation did not flow in a proper way through the models. So created this new method.

In [ ]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()
#openai_client = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

gemini_client = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama_client = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
requests.get("http://localhost:11434/").content

!ollama pull llama3.2

In [ ]:
!ollama list

In [ ]:
#models and personalities

gpt_model = "gpt-4.1-mini"
gemini_model = "gemini-2.5-flash-lite"
ollama_model = "llama3.2:latest"

bots = {
    "Alex":{
        "model_type":"openai",
        "model":gpt_model,
        "personality":("You are chatbot Alex. You are rude, argumentative, and strongly opinionated. "
            "You are in a chat with chatbot Brad and chatbot Charlie. "
            "You try to disagree with almost everything the others say. "
            "You often counter their opinions and try to prove that you are correct.")

    },
    "Brad":{
        "model_type":"gemini",
        "model":gemini_model,
        "personality":("You are chatbot Brad. You are calm, polite, and diplomatic. "
            "You are in a chat with chatbot Alex and chatbot Charlie. "
            "You try to reduce conflict and find common ground.")
    },
    "Charlie":{
        "model_type":"ollama",
        "model":ollama_model,
        "personality":("You are chatbot Charlie. You are analytical, curious, and balanced. "
            "You are in a chat with chatbot Alex and chatbot Brad. "
            "You try to examine both sides carefully before giving your view.")
    }
}
  


In [ ]:
bot_order = ["Alex", "Brad", "Charlie"]

conversation = [
    {"speaker": "Alex", "content": "Hi there. Today we will be discussing Marvel movies, how the movies after Avengers Endgame are boring and totally not worth it "},
    {"speaker": "Brad", "content": "Hello Alex. Nice Topic."},
    {"speaker": "Charlie", "content": "Let us begin."}
]

In [ ]:
# keeping conversation together

def format_coversation(conversation):
    text = ""

    for message in conversation:
       
        text += f"{message['speaker']}: {message['content']}\n"
    return text


In [ ]:
#function to generate the prompt for each model

def build_prompt(bot_name,bot_data,conversation):
    conversation_text = format_coversation(conversation)

    prompt = (
        f"{bot_data['personality']}\n\n"
        f"Here is the conversation so far:\n "
        f"{conversation_text}\n"
        f"Now respond with your thoughts as chatbot{bot_name}."
        f"Keep your conversation short, natural and conversational." 
        f"Do not write the names of other bots before your response. "
        f"Only write your own next message."
    )

    return prompt

In [ ]:
def call_openai_model(model,prompt):
    messages = [
        {"role":"user","content":prompt}
        ]
    response = openai.chat.completions.create(
        model = model, messages = messages
    )
    return response.choices[0].message.content

In [ ]:
def call_gemini_model(model,prompt):
    messages = [
        {"role":"user","content":prompt}
        ]
    response = gemini_client.chat.completions.create(
        model = model, messages = messages
    )
    return response.choices[0].message.content

In [ ]:
def call_ollama_model(model,prompt):
    messages = [
        {"role":"user","content":prompt}
        ]
    response = ollama_client.chat.completions.create(
        model = model, messages = messages
    )
    return response.choices[0].message.content

In [ ]:
#call the bots

def call_bot(bot_name, bots, conversation):
    bot_data = bots[bot_name]

    prompt = build_prompt(bot_name, bot_data, conversation)

    if bot_data["model_type"] == "openai":
        response = call_openai_model(bot_data["model"], prompt)

    elif bot_data["model_type"] == "gemini":
        response = call_gemini_model(bot_data["model"], prompt)

    elif bot_data["model_type"] == "ollama":
        response = call_ollama_model(bot_data["model"], prompt)

    else:
        raise ValueError(f"Unknown model type: {bot_data['model_type']}")

    return response

In [ ]:
for i in conversation:
    # print(i['speaker'],": ",i['content'])
    m_text = f"**{i['speaker']}**: {i['content']}"
    display(Markdown(m_text))

#conversation

for i in range(5):
    for bot_name in bot_order:
        response = call_bot(bot_name, bots, conversation)

        display(Markdown(f"\n### {bot_name}"))

        display(Markdown(response))

        conversation.append({
            "speaker": bot_name,
            "content": response
        })